
E01 : Entraînez un modèle de langage trigramme, c'est-à-dire prenez deux caractères en entrée pour prédire le troisième. Vous pouvez utiliser un algorithme de comptage ou un réseau de neurones. Évaluez la fonction de perte : est-ce mieux qu'avec un modèle bigramme ?

E02 : Divisez aléatoirement l'ensemble de données en 80 % pour l'entraînement, 10 % pour le développement et 10 % pour les tests. Entraînez les modèles de bigrammes et de trigrammes uniquement sur l'ensemble d'entraînement. Évaluez-les sur les ensembles de développement et de test. Que constatez-vous ?

E03 : Utilisez l'ensemble de développement pour ajuster l'intensité du lissage (ou de la régularisation) du modèle de trigrammes ; autrement dit, testez plusieurs valeurs et déterminez la plus performante en fonction de la perte sur l'ensemble de développement. Quels schémas observez-vous dans les pertes sur les ensembles d'entraînement et de développement lors de cet ajustement ? Choisissez la meilleure valeur de lissage et évaluez-la sur l'ensemble de test, une fois et à la fin. Quelle est la performance obtenue ?

E04 : Nous avons constaté que nos vecteurs one-hot sélectionnent simplement une ligne de W ; leur production explicite semble donc superflue. Pourriez-vous supprimer l'utilisation de `F.one_hot` et utiliser directement l'indexation des lignes de W ?

E05 : Recherchez et utilisez `F.cross_entropy` à la place. Vous devriez obtenir le même résultat. Pouvez-vous expliquer pourquoi il est préférable d'utiliser `F.cross_entropy` ?

E06 : Méta-exercice ! Trouvez un exercice amusant et intéressant et réalisez-le.

In [1]:
import torch
import torch.nn.functional as F

In [2]:
g = torch.Generator().manual_seed(2147483647)

In [3]:
names = open("names.txt", "r").read().splitlines()
names[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
train, val, test = torch.utils.data.random_split(names, [int(0.8*len(names)), int(0.1*len(names)), len(names) - int(0.8*len(names)) - int(0.1*len(names))], generator=g)

In [5]:
27*27

729

In [6]:
stoi = {s:i+1 for i,s in enumerate(sorted(list(set("".join(names)))))}
stoi["."] = 0
itos = {i:s for s,i in stoi.items()}

In [7]:
def build_dataset(words):
    xs = []
    ys = []
    for w in words:
        chs = [".", "."] + list(w) + ["."]
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            ix3 = stoi[ch3]
            xs.append(ix1 * 27 + ix2)
            ys.append(ix3)
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    return xs, ys

In [ ]:
def training(words, step_size=300, learning_rate=50, regularization_strength=0.01):
    W = torch.randn((729, 27), generator=g, requires_grad=True)
    xs, ys = build_dataset(words)
    xenc = F.one_hot(xs, num_classes=729).float()
    for _ in range(step_size):
        logits = xenc @ W
        # counts = logits.exp()
        # probs = counts / counts.sum(1, keepdims=True)
        # loss = (
        #     -probs[torch.arange(ys.shape[0]), ys].log().mean()
        #     + regularization_strength * (W**2).mean()
        # )
        loss = F.cross_entropy(logits, ys) + regularization_strength * (W**2).mean()
        W.grad = None
        loss.backward()

        W.data += -learning_rate * W.grad
    return loss.item(), W

@torch.no_grad()
def evaluate(words, W, regularization_strength=0.01):
    xs, ys = build_dataset(words)
    xenc = F.one_hot(xs, num_classes=729).float()
    logits = xenc @ W
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = (
    #    -probs[torch.arange(ys.shape[0]), ys].log().mean()
    # )
    #return loss.item()
    loss = F.cross_entropy(logits, ys)
    return loss.item()

In [9]:
reg_strength = 0.1
min_loss = float("inf")
for lr in [10, 50, 100, 200]:
    loss, W = training(
        train, step_size=300, learning_rate=lr, regularization_strength=reg_strength
    )
    val_loss = evaluate(val, W, regularization_strength=reg_strength)
    print(f"lr: {lr}, training loss: {loss}, validation loss: {val_loss}")
    if val_loss < min_loss:
        min_loss, best_lr, best_W = val_loss, lr, W

print(f"Best learning rate: {best_lr}, Best validation loss: {min_loss}, with traning loss: {loss}")

print("Test loss:", evaluate(test, best_W, regularization_strength=reg_strength))

lr: 10, training loss: 2.715484857559204, validation loss: 2.63590407371521
lr: 50, training loss: 2.4120473861694336, validation loss: 2.342851161956787
lr: 100, training loss: 2.351175308227539, validation loss: 2.292264938354492
lr: 200, training loss: 2.316955089569092, validation loss: 2.2685787677764893
Best learning rate: 200, Best validation loss: 2.2685787677764893, with traning loss: 2.316955089569092
Test loss: 2.2611281871795654


In [10]:
for _ in range(20):
     ix1, ix2 = 0, 0

     while True:
          ix = ix1 * 27 + ix2
          x_enc = F.one_hot(torch.tensor([ix]), num_classes=729).float()
          logits = x_enc @ W
          counts = logits.exp()
          probs = counts / counts.sum(1, keepdims=True)
          ix3 = torch.multinomial(probs, num_samples=1, generator=g).item()
          if ix3 == 0:
               break
          ix1 = ix2
          ix2 = ix3
          print(itos[ix3], end="")
     print()

mirasechah
dees
ystercmjkki
kiyusteslynni
seb
denle
basiarsymir
coleon
lachintrann
avelzjbmjchuthal
saceluomaradelaydenna
lo
anaysmah
gjzhqvgjyqhya
treyah
kaock
quewpsnlykennaarie
sxpcgxtdb
lutej
tata


In [11]:
W

tensor([[-4.1617,  2.1425,  0.6366,  ..., -1.5895, -0.2905,  0.2979],
        [-2.4759,  0.9589,  0.9042,  ..., -1.0720,  0.7638,  0.6256],
        [-0.5707,  3.3359, -0.6725,  ..., -0.6124, -0.1612, -0.7339],
        ...,
        [ 0.1053,  1.4903, -0.3123,  ...,  0.1533,  0.5749,  0.3504],
        [ 1.8645,  1.7687, -1.0593,  ..., -0.6040, -0.5708, -0.7974],
        [ 0.5971,  0.7584, -0.1060,  ..., -0.2975,  0.5827, -1.0036]],
       requires_grad=True)